# Project FORESIGHT — D1 Data Pipeline

**Client:** NorthBay Living  
**Project:** Demand & Inventory Intelligence  
**Deliverable:** D1 — Reproducible Data Pipeline

This notebook ingests, profiles, cleans and validates the provided Online Retail II transaction dataset, then creates analysis-ready datasets for the FORESIGHT workflow.

## D1 Objectives
- Combine both transaction sheets
- Profile data quality
- Handle missing values and duplicates
- Standardize data types
- Identify cancellations/returns and invalid transactions
- Create clean transaction-level sales data
- Create `sales_daily`
- Create `sku_master`
- Create a derived `calendar`
- Document the absence of actual inventory data
- Validate and export processed datasets

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Upload the Excel Dataset

In [2]:
# from google.colab import files

# uploaded = files.upload()
# file_name = list(uploaded.keys())[0]

# print("Uploaded file:", file_name)

## 3. Load Both Excel Sheets

In [3]:
excel_file = pd.ExcelFile("../data/online_retail_II.xlsx")

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['Year 2009-2010', 'Year 2010-2011']


In [4]:
file_name = "../data/online_retail_II.xlsx"
df_2009_2010 = pd.read_excel(file_name, sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel(file_name, sheet_name="Year 2010-2011")

print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)

2009-2010 shape: (525462, 8)
2010-2011 shape: (541910, 8)


## 4. Combine Both Years

In [5]:
df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)
display(df.head())

Combined dataset shape: (1067372, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 5. Initial Data Profiling

In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())

Rows: 1067372
Columns: 8

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067372 entries, 0 to 1067371
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  float64       
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067372 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 65.1+ MB


In [8]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,1067371.0,53628.0,537434.0,1350.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,1067371,5305,85123A,5829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,1062989,5698,WHITE HANGING HEART T-LIGHT HOLDER,5918,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,1067371.0,NaN,NaN,NaN,9.938898,-80995.0,1.0,3.0,10.0,80995.0,172.705794
InvoiceDate,1067371,NaN,NaN,NaN,2011-01-02 21:13:55.394028544,2009-12-01 07:45:00,2010-07-09 09:46:00,2010-12-07 15:28:00,2011-07-22 10:23:00,2011-12-09 12:50:00,NaN
Price,1067372.0,NaN,NaN,NaN,4.649388,-53594.36,1.25,2.1,4.15,38970.0,123.553001
Customer ID,824364.0,NaN,NaN,NaN,15324.638504,12346.0,13975.0,15255.0,16797.0,18287.0,1697.46445
Country,1067371,43,United Kingdom,981330,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()} unique values")

Invoice: 53628 unique values
StockCode: 5305 unique values
Description: 5698 unique values
Quantity: 1057 unique values
InvoiceDate: 47635 unique values
Price: 2808 unique values
Customer ID: 5942 unique values
Country: 43 unique values


## 6. Data Cleaning

### 6.1 Rename Columns

In [10]:
df = df.rename(columns={
    "Invoice": "invoice_id",
    "StockCode": "sku_id",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
})

display(df.head())

,invoice_id,sku_id,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 6.2 Missing-Value Assessment

In [11]:
missing_values = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(4)
}).sort_values("missing_count", ascending=False)

display(missing_values)

,missing_count,missing_percentage
customer_id,243008,22.7669
description,4383,0.4106
invoice_id,1,0.0001
sku_id,1,0.0001
quantity,1,0.0001
invoice_date,1,0.0001
country,1,0.0001
unit_price,0,0.0000


**Cleaning decision:** `customer_id` is not required for SKU-level demand forecasting, so its missing values are retained. Missing descriptions are filled with `Unknown`. Rows missing core transaction fields such as invoice ID, SKU, quantity, invoice date, or country are excluded from the sales dataset.

In [12]:
df["invoice_id"] = df["invoice_id"].astype("string").str.strip()
df["sku_id"] = df["sku_id"].astype("string").str.strip()
df["description"] = df["description"].astype("string").str.strip()
df["customer_id"] = df["customer_id"].astype("string").str.strip()
df["country"] = df["country"].astype("string").str.strip()

df["description"] = df["description"].fillna("Unknown")

print("Missing values after description handling:")
display(df.isna().sum())

Missing values after description handling:


invoice_id           1
sku_id               1
description          0
quantity             1
invoice_date         1
unit_price           0
customer_id     243008
country              1
dtype: int64

### 6.3 Remove Exact Duplicate Records

In [13]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows before removal:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)
print("Duplicate rows after removal:", df.duplicated().sum())

Duplicate rows before removal: 34335
Shape after removing duplicates: (1033037, 8)
Duplicate rows after removal: 0


### 6.4 Fix Data Types

In [14]:
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

print(df.dtypes)

invoice_id      string[python]
sku_id          string[python]
description     string[python]
quantity               float64
invoice_date    datetime64[ns]
unit_price             float64
customer_id     string[python]
country         string[python]
dtype: object


### 6.5 Identify Cancelled Invoices / Returns

In [15]:
df["is_cancelled"] = (
    df["invoice_id"].str.upper().str.startswith("C", na=False)
)

print("Cancelled rows:", int(df["is_cancelled"].sum()))
print("Non-cancelled rows:", int((~df["is_cancelled"]).sum()))

display(df[df["is_cancelled"]].head())

Cancelled rows: 19104
Non-cancelled rows: 1013933


,invoice_id,sku_id,description,quantity,invoice_date,unit_price,customer_id,country,is_cancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12.0,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6.0,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4.0,2009-12-01 10:33:00,4.25,16321.0,Australia,True
181,C489449,21896,POTTING SHED TWINE,-6.0,2009-12-01 10:33:00,2.10,16321.0,Australia,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12.0,2009-12-01 10:33:00,2.95,16321.0,Australia,True


### 6.6 Check Invalid Quantities and Prices

In [16]:
print("Missing quantity:", df["quantity"].isna().sum())
print("Zero quantity:", (df["quantity"] == 0).sum())
print("Negative quantity:", (df["quantity"] < 0).sum())

print("\nMissing price:", df["unit_price"].isna().sum())
print("Zero price:", (df["unit_price"] == 0).sum())
print("Negative price:", (df["unit_price"] < 0).sum())

Missing quantity: 1
Zero quantity: 0
Negative quantity: 22496

Missing price: 0
Zero price: 6014
Negative price: 5


### 6.7 Create Clean Sales Dataset

For demand forecasting, retain completed positive-sale transactions only. Cancellation/return rows are excluded from the demand dataset rather than being treated as positive demand.

In [17]:
sales_df = df[
    (~df["is_cancelled"]) &
    (df["quantity"] > 0) &
    (df["unit_price"] > 0) &
    (df["invoice_id"].notna()) &
    (df["sku_id"].notna()) &
    (df["invoice_date"].notna()) &
    (df["country"].notna())
].copy()

sales_df["revenue"] = sales_df["quantity"] * sales_df["unit_price"]
sales_df["date"] = sales_df["invoice_date"].dt.normalize()

print("Rows after duplicate removal:", len(df))
print("Clean sales rows:", len(sales_df))
display(sales_df.head())

Rows after duplicate removal: 1033037
Clean sales rows: 1007913


,invoice_id,sku_id,description,quantity,invoice_date,unit_price,customer_id,country,is_cancelled,revenue,date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,2009-12-01


### 6.8 Validate Clean Sales Data

In [18]:
validation = {
    "missing_invoice_id": int(sales_df["invoice_id"].isna().sum()),
    "missing_sku_id": int(sales_df["sku_id"].isna().sum()),
    "missing_invoice_date": int(sales_df["invoice_date"].isna().sum()),
    "non_positive_quantity": int((sales_df["quantity"] <= 0).sum()),
    "non_positive_price": int((sales_df["unit_price"] <= 0).sum()),
    "duplicate_rows": int(sales_df.duplicated().sum())
}

display(pd.DataFrame([validation]))

assert all(value == 0 for value in validation.values())

print("Clean sales validation PASSED.")

,missing_invoice_id,missing_sku_id,missing_invoice_date,non_positive_quantity,non_positive_price,duplicate_rows
0,0,0,0,0,0,0


Clean sales validation PASSED.


## 7. Create Analysis-Ready Datasets

### 7.1 Create `sales_daily`

FORESIGHT requires daily SKU-level sales data. The transaction-level data is aggregated to one row per `date + sku_id`.

In [19]:
sales_daily = (
    sales_df
    .groupby(["date", "sku_id"], as_index=False)
    .agg(
        units_sold=("quantity", "sum"),
        revenue=("revenue", "sum"),
        unit_price=("unit_price", "mean")
    )
)

print("sales_daily shape:", sales_daily.shape)
print("Unique SKUs:", sales_daily["sku_id"].nunique())
print("Unique dates:", sales_daily["date"].nunique())
print("Duplicate SKU-day records:",
      sales_daily.duplicated(["date", "sku_id"]).sum())

display(sales_daily.head(10))

sales_daily shape: (532838, 5)
Unique SKUs: 4916
Unique dates: 604
Duplicate SKU-day records: 0


,date,sku_id,units_sold,revenue,unit_price
0,2009-12-01,10002,12.0,10.20,0.850
1,2009-12-01,10120,60.0,12.60,0.210
2,2009-12-01,10123C,3.0,3.90,1.300
3,2009-12-01,10123G,2.0,3.40,1.700
4,2009-12-01,10125,5.0,5.10,1.275
5,2009-12-01,10133,6.0,5.10,0.850
6,2009-12-01,10135,17.0,21.25,1.250
7,2009-12-01,11001,2.0,6.86,3.430
8,2009-12-01,15034,3.0,1.11,0.485
9,2009-12-01,15036,55.0,35.75,0.650


### 7.2 Create `sku_master`

Only product information available in the source is derived. Category, unit cost and list price are not fabricated.

In [20]:
sku_master = (
    sales_df
    .groupby("sku_id", as_index=False)
    .agg(
        description=("description", "first"),
        first_sale_date=("date", "min"),
        last_sale_date=("date", "max"),
        average_selling_price=("unit_price", "mean"),
        total_units_sold=("quantity", "sum"),
        total_revenue=("revenue", "sum")
    )
)

print("sku_master shape:", sku_master.shape)
print("Duplicate SKU IDs:", sku_master["sku_id"].duplicated().sum())

display(sku_master.head(10))

sku_master shape: (4916, 7)
Duplicate SKU IDs: 0


,sku_id,description,first_sale_date,last_sale_date,average_selling_price,total_units_sold,total_revenue
0,10002,INFLATABLE POLITICAL GLOBE,2009-12-01,2011-04-18,0.979155,8671.0,6942.26
1,10002R,ROBOT PENCIL SHARPNER,2009-12-02,2010-01-25,5.133333,4.0,20.57
2,10080,GROOVY CACTUS INFLATABLE,2009-12-02,2011-11-21,0.505000,315.0,129.29
3,10109,BENDY COLOUR PENCILS,2009-12-03,2009-12-03,0.420000,4.0,1.68
4,10120,DOGGY RUBBER,2009-12-01,2011-12-04,0.240137,664.0,142.74
5,10123C,HEARTS WRAPPING TAPE,2009-12-01,2011-03-31,0.790635,650.0,254.41
6,10123G,ARMY CAMO WRAPPING TAPE,2009-12-01,2010-11-29,0.820588,2251.0,165.56
7,10124A,SPOTS ON RED BOOKCOVER TAPE,2010-03-19,2011-11-06,0.420000,58.0,24.36
8,10124G,ARMY CAMO BOOKCOVER TAPE,2010-05-13,2011-11-06,0.462000,33.0,14.28
9,10125,MINI FUNKY DESIGN TAPES,2009-12-01,2011-12-09,0.930636,2103.0,1715.78


### 7.3 Create Derived `calendar`

The source does not contain the specified calendar table, so calendar attributes are derived from transaction dates.

In [21]:
calendar = pd.DataFrame({
    "date": pd.date_range(
        start=sales_df["date"].min(),
        end=sales_df["date"].max(),
        freq="D"
    )
})

calendar["year"] = calendar["date"].dt.year
calendar["month"] = calendar["date"].dt.month
calendar["month_name"] = calendar["date"].dt.month_name()
calendar["week"] = calendar["date"].dt.isocalendar().week.astype(int)
calendar["day_of_week"] = calendar["date"].dt.dayofweek
calendar["day_name"] = calendar["date"].dt.day_name()
calendar["quarter"] = calendar["date"].dt.quarter
calendar["is_weekend"] = (calendar["day_of_week"] >= 5).astype(int)

display(calendar.head())
print("Calendar rows:", len(calendar))

,date,year,month,month_name,week,day_of_week,day_name,quarter,is_weekend
0,2009-12-01,2009,12,December,49,1,Tuesday,4,0
1,2009-12-02,2009,12,December,49,2,Wednesday,4,0
2,2009-12-03,2009,12,December,49,3,Thursday,4,0
3,2009-12-04,2009,12,December,49,4,Friday,4,0
4,2009-12-05,2009,12,December,49,5,Saturday,4,1


Calendar rows: 739


## 8. Inventory Data Limitation

The supplied Online Retail II dataset does not contain observed inventory snapshots, on-hand units, on-order units, lead times, or reorder points.

These fields cannot be reconstructed without fabricating data. Therefore, no fake `inventory_snapshots` table is created. Any inventory-dependent risk analysis in D4 will be explicitly labelled as a proxy/assumption or limitation.

In [22]:
inventory_available = False

inventory_fields = [
    "on_hand_units",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]

print("Observed inventory data available:", inventory_available)
print("Required inventory fields:", inventory_fields)

Observed inventory data available: False
Required inventory fields: ['on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point']


## 9. Final D1 Data Quality Report

In [23]:
data_quality_report = pd.DataFrame({
    "metric": [
        "Raw rows",
        "Raw columns",
        "Duplicate rows removed",
        "Missing customer IDs",
        "Descriptions filled as Unknown",
        "Cancelled rows",
        "Clean sales rows",
        "Unique SKUs",
        "Unique sales dates",
        "sales_daily rows",
        "sku_master rows",
        "calendar rows",
        "Inventory table available"
    ],
    "value": [
        len(df_2009_2010) + len(df_2010_2011),
        len(df.columns),
        duplicate_count,
        int(df["customer_id"].isna().sum()),
        int((df["description"] == "Unknown").sum()),
        int(df["is_cancelled"].sum()),
        len(sales_df),
        sales_df["sku_id"].nunique(),
        sales_df["date"].nunique(),
        len(sales_daily),
        len(sku_master),
        len(calendar),
        inventory_available
    ]
})

display(data_quality_report)

,metric,value
0,Raw rows,1067372
1,Raw columns,9
2,Duplicate rows removed,34335
3,Missing customer IDs,235152
4,Descriptions filled as Unknown,4276
5,Cancelled rows,19104
6,Clean sales rows,1007913
7,Unique SKUs,4916
8,Unique sales dates,604
9,sales_daily rows,532838


## 10. Export Processed Data

In [24]:
os.makedirs("../data/foresight_data/processed", exist_ok=True)
os.makedirs("../data/foresight_reports", exist_ok=True)

sales_df.to_csv(
    "../data/foresight_data/processed/clean_transactions.csv",
    index=False
)

sales_daily.to_csv(
    "../data/foresight_data/processed/sales_daily.csv",
    index=False
)

sku_master.to_csv(
    "../data/foresight_data/processed/sku_master.csv",
    index=False
)

calendar.to_csv(
    "../data/foresight_data/processed/calendar.csv",
    index=False
)

data_quality_report.to_csv(
    "../data/foresight_reports/data_quality_report.csv",
    index=False
)

print("Processed datasets and data-quality report saved successfully.")

Processed datasets and data-quality report saved successfully.


## 11. Final D1 Validation

In [25]:
assert sales_daily["date"].notna().all()
assert sales_daily["sku_id"].notna().all()
assert (sales_daily["units_sold"] > 0).all()
assert (sales_daily["revenue"] > 0).all()
assert sales_daily.duplicated(["date", "sku_id"]).sum() == 0
assert sku_master["sku_id"].is_unique
assert calendar["date"].is_unique

print("All D1 validation checks PASSED.")

All D1 validation checks PASSED.


## 12. D1 Summary

### Completed
- Data ingestion from both Excel sheets
- Combined 2009–2010 and 2010–2011 records
- Data profiling
- Missing-value assessment
- Duplicate removal
- Data-type standardization
- Cancellation/return identification
- Invalid transaction filtering for the demand dataset
- Revenue calculation
- Daily SKU-level `sales_daily`
- `sku_master`
- Derived `calendar`
- Data-quality report
- Validation
- CSV exports

### Limitation
Actual inventory snapshots are not present in the supplied Online Retail II dataset. No inventory values are fabricated.

### D1 Status
**COMPLETE — ready for D2 EDA and baseline analysis.**